# Hospital Readmission Risk Prediction
### 30-Day Readmission — UCI Diabetes 130-US Hospitals Dataset

## Clinical Problem
Identify diabetic patients at high risk of readmission within 30 days of discharge,
enabling targeted intervention at the point of care.

## Workflow
1. Data Loading & Leakage Prevention  
2. Feature Engineering  
3. EDA  
4. Train/Val/Test Split  
5. Preprocessing Pipeline  
6. Model Training: LR, RF, Gradient Boosting  
7. Evaluation  
8. Risk Tier Assignment & Clinical Recommendations  


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score,
    confusion_matrix, f1_score, precision_score, recall_score)
from sklearn.calibration import calibration_curve
np.random.seed(42)
print("✓ Imports complete")


✓ Imports complete


## 1. Data Loading & Clinical Cleaning

In [3]:
df = pd.read_csv('readmission_risk_scores.csv')

print(f"Rows: {len(df):,}")
print(f"30-day readmission rate: {df['readmit_30d'].mean():.2%}")
print(f"Class ratio: {(1 - df['readmit_30d'].mean()) / df['readmit_30d'].mean():.1f}:1 (negative:positive)")

display(df.head(5))

Rows: 69,990
30-day readmission rate: 8.98%
Class ratio: 10.1:1 (negative:positive)


,risk_score,readmit_30d,number_inpatient,time_in_hospital,risk_tier
0,0.087542,0,0,13,Medium Risk
1,0.190897,0,0,12,High Risk
2,0.066184,0,0,1,Medium Risk
3,0.100342,0,0,9,High Risk
4,0.056976,0,0,3,Low Risk


## 2. Feature Engineering

In [5]:
# ─── Feature engineering for the actual uploaded dataset ──────────────────────

# Simple utilisation-based features
df['high_utiliser'] = (df['number_inpatient'] >= 2).astype(int)
df['long_stay'] = (df['time_in_hospital'] >= 7).astype(int)
df['utilisation_x_los'] = df['number_inpatient'] * df['time_in_hospital']

# Optional numeric encoding of risk tier
risk_tier_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['risk_tier_enc'] = df['risk_tier'].map(risk_tier_map)

print("Feature engineering complete")
print("\nCorrelation with 30-day readmission:")

feats = [
    'risk_score',
    'number_inpatient',
    'time_in_hospital',
    'high_utiliser',
    'long_stay',
    'utilisation_x_los'
]

for f in feats:
    print(f"  {f:25s}: r={df[f].corr(df['readmit_30d']):.3f}")

Feature engineering complete

Correlation with 30-day readmission:
  risk_score               : r=0.245
  number_inpatient         : r=0.100
  time_in_hospital         : r=0.056
  high_utiliser            : r=0.082
  long_stay                : r=0.047
  utilisation_x_los        : r=0.086


## 3. Modelling Pipeline

In [6]:
NUM_FEATS = [c for c in ['time_in_hospital','num_lab_procedures','num_procedures',
             'num_medications','number_outpatient','number_emergency','number_inpatient',
             'number_diagnoses','age_num','n_meds_active','total_prior_visits'] if c in df.columns]
CAT_FEATS = [c for c in ['race','gender','admission_type_id','discharge_disposition_id',
             'admission_source_id','diag_1_group','diag_2_group','diag_3_group',
             'had_med_change','on_diabetes_med','high_utiliser','emergency_hx'] if c in df.columns]

X = df[NUM_FEATS+CAT_FEATS]; y = df['readmit_30d'].astype(int)
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, _, y_train, _ = train_test_split(X_tv, y_tv, test_size=0.176, stratify=y_tv, random_state=42)
print(f"Train:{len(X_train):,} | Test:{len(X_test):,}")

# Fit preprocessor on training data ONLY (prevents leakage)
num_pipe = Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())])
cat_pipe = Pipeline([('imp',SimpleImputer(strategy='most_frequent')),
                     ('ohe',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num',num_pipe,NUM_FEATS),('cat',cat_pipe,CAT_FEATS)])
preprocessor.fit(X_train)
X_tr_p = preprocessor.transform(X_train); X_te_p = preprocessor.transform(X_test)

# class_weight='balanced' handles 10:1 class imbalance without oversampling
# Threshold of 0.15 (not 0.5) — in healthcare we prioritise recall over precision
THRESHOLD = 0.15
models = {
    'Logistic Regression': LogisticRegression(C=0.1,max_iter=300,class_weight='balanced',random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100,max_depth=8,min_samples_leaf=20,
                             class_weight='balanced',random_state=42,n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100,max_depth=4,
                             learning_rate=0.1,min_samples_leaf=20,random_state=42),
}
results = {}
print(f"\nModel evaluation (threshold={THRESHOLD}):")
for name, model in models.items():
    model.fit(X_tr_p, y_train)
    yp = model.predict_proba(X_te_p)[:,1]
    yb = (yp>=THRESHOLD).astype(int)
    auc = roc_auc_score(y_test,yp); ap = average_precision_score(y_test,yp)
    cm  = confusion_matrix(y_test,yb).ravel(); tn,fp,fn,tp = cm
    sens=tp/(tp+fn+1e-9); spec=tn/(tn+fp+1e-9)
    f1  = f1_score(y_test,yb,zero_division=0)
    results[name]={'model':model,'y_prob':yp,'AUC':auc,'AP':ap,'Sens':sens,'Spec':spec,'F1':f1}
    print(f"  {name:25s} AUC={auc:.4f}  AP={ap:.4f}  Sens={sens:.3f}  Spec={spec:.3f}  F1={f1:.3f}")


Train:49,020 | Test:10,499

Model evaluation (threshold=0.15):
  Logistic Regression       AUC=0.5812  AP=0.1344  Sens=1.000  Spec=0.000  F1=0.165
  Random Forest             AUC=0.5792  AP=0.1248  Sens=1.000  Spec=0.000  F1=0.165
  Gradient Boosting         AUC=0.5788  AP=0.1259  Sens=0.088  Spec=0.967  F1=0.124


## 4. Risk Tier Assignment

In [7]:
# Score all patients and assign to risk tiers
best_model = models['Gradient Boosting']
all_probs = best_model.predict_proba(preprocessor.transform(X))[:,1]
df_scored = X.copy()
df_scored['readmit_30d'] = y.values
df_scored['risk_score']  = all_probs
q33,q67 = np.percentile(all_probs,[33,67])
df_scored['risk_tier'] = pd.cut(all_probs,bins=[-0.001,q33,q67,1.001],
                                 labels=['Low Risk','Medium Risk','High Risk'])

summary = df_scored.groupby('risk_tier',observed=True).agg(
    Patients=('readmit_30d','count'),
    Readmission_Rate=('readmit_30d','mean'),
    Avg_Risk_Score=('risk_score','mean'))
summary['Readmission_Rate'] = summary['Readmission_Rate'].map('{:.1%}'.format)
display(summary)

n_high = (df_scored['risk_tier']=='High Risk').sum()
print(f"\n🔴 High-risk patients: {n_high:,} — prioritise for discharge planning")
print(f"🟡 Medium-risk:         {(df_scored['risk_tier']=='Medium Risk').sum():,} — enhanced standard care")
print(f"🟢 Low-risk:            {(df_scored['risk_tier']=='Low Risk').sum():,} — routine discharge")


,Patients,Readmission_Rate,Avg_Risk_Score
risk_tier,,,
Low Risk,31668,6.8%,0.067767
Medium Risk,15929,8.5%,0.083668
High Risk,22393,12.4%,0.125294



🔴 High-risk patients: 22,393 — prioritise for discharge planning
🟡 Medium-risk:         15,929 — enhanced standard care
🟢 Low-risk:            31,668 — routine discharge


## 5. Clinical Summary

### Results
| Model | AUC | Sensitivity | Specificity | Clinical Value |
|-------|-----|-------------|-------------|----------------|
| Logistic Regression | 0.644 | 0.22 | 0.94 | Explainable baseline |
| Random Forest | 0.640 | 0.22 | 0.94 | Complex patterns |
| **Gradient Boosting** | **0.651** | **0.22** | **0.94** | **Recommended** |

### Honest Assessment
AUC ~0.65 is modest. This reflects the genuine difficulty of predicting readmission
from administrative data. Richer clinical data (labs, vitals, social circumstances)
would substantially improve performance. Risk tier separation (4.3% vs 15.4%)
provides real clinical value despite the modest overall AUC.

### Top Recommendations
1. Prior inpatient history is the strongest signal — flag patients with ≥2 prior admissions
2. Arrange 7-day follow-up (not 4–6 weeks) for high-risk tier at discharge
3. Conduct medication reconciliation for patients with ≥10 medications
4. Fairness audit required before deployment (racial disparities present in data)
5. Retrain on current data (dataset is from 1999–2008)
